In [1]:
import pandas as pd
import pyarrow.dataset as ds
import numpy as np
from pathlib import Path

In [ ]:
path = Path.cwd()
files_path = path.parent / 'data' / 'raw'

dataset = ds.dataset(files_path, format='csv')
df = (dataset.to_table().to_pandas(self_destruct=True, types_mapper=pd.ArrowDtype))

print(f'Total de registros: {len(df)}')
df.head()

Total de registros:47248845


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,2,2015-01-15 19:05:39,2015-01-15 19:23:42,1,1.59,-73.993896,40.750111,1,N,-73.974785,40.750618,1,12.0,1.0,0.5,3.25,0.0,0.3,17.05
1,1,2015-01-10 20:33:38,2015-01-10 20:53:28,1,3.3,-74.001648,40.724243,1,N,-73.994415,40.759109,1,14.5,0.5,0.5,2.0,0.0,0.3,17.8
2,1,2015-01-10 20:33:38,2015-01-10 20:43:41,1,1.8,-73.963341,40.802788,1,N,-73.95182,40.824413,2,9.5,0.5,0.5,0.0,0.0,0.3,10.8
3,1,2015-01-10 20:33:39,2015-01-10 20:35:31,1,0.5,-74.009087,40.713818,1,N,-74.004326,40.719986,2,3.5,0.5,0.5,0.0,0.0,0.3,4.8
4,1,2015-01-10 20:33:39,2015-01-10 20:52:58,1,3.0,-73.971176,40.762428,1,N,-74.004181,40.742653,2,15.0,0.5,0.5,0.0,0.0,0.3,16.3


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47248845 entries, 0 to 47248844
Data columns (total 19 columns):
 #   Column                 Dtype                
---  ------                 -----                
 0   VendorID               int64[pyarrow]       
 1   tpep_pickup_datetime   timestamp[s][pyarrow]
 2   tpep_dropoff_datetime  timestamp[s][pyarrow]
 3   passenger_count        int64[pyarrow]       
 4   trip_distance          double[pyarrow]      
 5   pickup_longitude       double[pyarrow]      
 6   pickup_latitude        double[pyarrow]      
 7   RateCodeID             int64[pyarrow]       
 8   store_and_fwd_flag     string[pyarrow]      
 9   dropoff_longitude      double[pyarrow]      
 10  dropoff_latitude       double[pyarrow]      
 11  payment_type           int64[pyarrow]       
 12  fare_amount            double[pyarrow]      
 13  extra                  double[pyarrow]      
 14  mta_tax                double[pyarrow]      
 15  tip_amount             double[

In [4]:
# Verifificação de nulos
count_null = df.isnull().sum()
if count_null.sum() > 0:
    print(f'Valores nulos:\n{count_null[count_null > 0]}')
else:
    print('Nenhum valor nulo encontrado no dataframe')

Valores nulos:
RateCodeID               34499859
improvement_surcharge           3
dtype: int64


In [5]:
# Detecção de anomalias
# Distância inválida
invalid_distance = (df['trip_distance'] <= 0) | (df['trip_distance'] > 100)

# Duração invalida
# Coluna de duração da corrida (min)
df['duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds()/60
invalid_duration = (df['duration_min'] <= 0) | (df['duration_min'] > 240)

# Quantidade de passageiros invalida
invalid_passengers = (df['passenger_count'] <= 0) | (df['passenger_count'] > 6)

# Tarifa invalida
invalid_fare = df['fare_amount'] < 0

# Coordenadas fora de NYC
invalid_coords = (
    (df['pickup_latitude'] < 40.4) | (df['pickup_latitude'] > 41.0) |
    (df['dropoff_latitude'] < 40.4) | (df['dropoff_latitude'] > 41.0) |
    (df['pickup_longitude'] < -74.3) | (df['pickup_longitude'] > -73.7) |
    (df['dropoff_longitude'] < -74.3) | (df['dropoff_longitude'] > -73.7)
)

total = len(df)

print(f"Distância inválida (<=0 ou >100mi): {invalid_distance.sum():,} ({invalid_distance.sum()/total*100:.2f}%)")
print(f"Duração inválida (<=0 ou >240):     {invalid_duration.sum():,} ({invalid_duration.sum()/total*100:.7f}%)")
print(f"Passageiros inválidos (<=0 ou >6):  {invalid_passengers.sum():,} ({invalid_passengers.sum()/total*100:.7f}%)")
print(f"Tarifa negativa:                    {invalid_fare.sum():,} ({invalid_fare.sum()/total*100:.4f}%)")
print(f"Coordenadas fora de NYC:            {invalid_coords.sum():,} ({invalid_coords.sum()/total*100:.2f}%)")

Distância inválida (<=0 ou >100mi): 282,698 (0.60%)
Duração inválida (<=0 ou >240):     113,624 (0.2404800%)
Passageiros inválidos (<=0 ou >6):  8,441 (0.0178650%)
Tarifa negativa:                    17,153 (0.0363%)
Coordenadas fora de NYC:            870,414 (1.84%)


In [6]:
# Validações extras
# Consistência financeira
sum_taxes =  (df['fare_amount'] + df['extra'] + df['mta_tax'] + df['tip_amount'] + df['tolls_amount'] + df['improvement_surcharge'])
invalid_finance = df['total_amount'].round(2) != sum_taxes.round(2)

# Corridas repetidas
duplicated = df.duplicated(subset=['tpep_pickup_datetime', 'VendorID', 'trip_distance', 'total_amount'])

# Velocidade invalida
df['speed_mph'] = df['trip_distance'] / (df['duration_min'] / 60)
invalid_speed = (df['speed_mph'] <= 0) | (df['speed_mph'] > 80)

# Tipo de pagamento invalido
invalid_payment_type = ~df['payment_type'].isin([1, 2, 3, 4, 5, 6])

# VendorID invalido
invalid_vendor_ID = ~df['VendorID'].isin([1, 2])

# Integridade Temporal
pickup_order = (df['tpep_pickup_datetime'] <= df['tpep_dropoff_datetime'])
year_2015 = (df['tpep_pickup_datetime'].dt.year == 2015) & (df['tpep_pickup_datetime'].dt.month == 1)
year_2016 = (df['tpep_pickup_datetime'].dt.year == 2016) & (df['tpep_pickup_datetime'].dt.month.isin([1, 2, 3]))
invalid_time = ~(pickup_order & (year_2015 | year_2016))

total = len(df)

print(f'Inconsistencias financeiras: {invalid_finance.sum():,} ({invalid_finance.sum()/total*100:.2f}%)')
print(f'Corridas duplicadas: {duplicated.sum():,} ({duplicated.sum()/total*100:.4f}%)')
print(f'Velocidade invalida (<=0 ou >80): {duplicated.sum():,} ({duplicated.sum()/total*100:.4f}%)')
print(f'Tipo de pagamento invalido: {invalid_payment_type.sum():,} ({invalid_payment_type.sum()/total*100:.2f}%)')
print(f'Vendor ID invalido: {invalid_payment_type.sum():,} ({invalid_payment_type.sum()/total*100:.2f}%)')
print(f'Integridade Temporal invalida: {invalid_time.sum():,} ({invalid_time.sum()/total*100:.7f}%)')

Inconsistencias financeiras: 2,347,300 (4.97%)
Corridas duplicadas: 40,505 (0.0857%)
Velocidade invalida (<=0 ou >80): 40,505 (0.0857%)
Tipo de pagamento invalido: 0 (0.00%)
Vendor ID invalido: 0 (0.00%)
Integridade Temporal invalida: 473 (0.0010011%)


In [7]:
# Removendo os registros invalidos
all_invalid = (
    invalid_distance |
    invalid_duration | 
    invalid_passengers |
    invalid_fare |
    invalid_coords |
    invalid_finance | 
    duplicated | 
    invalid_speed | 
    invalid_payment_type |
    invalid_vendor_ID |  
    invalid_time
)

df_clean = df[~all_invalid]

clean_count = len(df_clean)

quality_score = (clean_count / total) * 100

print(f'Registros raw: {total:,}')
print(f'Registros clean: {clean_count:,}')
print(f'Score de qualidade: {quality_score:.2f}%')

Registros raw: 47,248,845
Registros clean: 43,774,384
Score de qualidade: 92.65%


In [8]:
import gc

del df
del all_invalid

gc.collect()

40

In [ ]:
# Padronizações
# Datetime
df_clean['pickup_datetime'] = pd.to_datetime(df_clean['tpep_pickup_datetime'])
df_clean['dropoff_datetime'] = pd.to_datetime(df_clean['tpep_dropoff_datetime'])

# Coluna velocidade (tratamento divisão por 0)
hours = df_clean['duration_min'] / 60
df_clean['speed_mph'] = (
    df_clean['trip_distance'] / hours.replace(0, np.nan)
).fillna(0)

# Novas colunas de tempo
df_clean['date'] = df_clean['pickup_datetime'].dt.date.astype(str)
df_clean['hour_of_day'] = df_clean['pickup_datetime'].dt.hour
df_clean['day_of_week'] = df_clean['pickup_datetime'].dt.dayofweek
df_clean['is_weekend'] = df_clean['day_of_week'].isin([5, 6])

# Nova coluna de tipo de pagamento
payment_labels = {1: 'Credit Card', 2: 'Cash', 3: 'No Charge', 4: 'Dispute', 5: 'Unknown', 6: 'Voided'}
df_clean['payment_label'] = df_clean['payment_type'].map(payment_labels).fillna('Other')

# Novas colunas rentabilidade (tratamento divisão por 0)
df_clean['revenue_per_mile'] = (
    df_clean['total_amount'] / df_clean['trip_distance'].replace(0, np.nan)
).fillna(0)

df_clean['revenue_per_min'] = (
    df_clean['total_amount'] / df_clean['duration_min'].replace(0, np.nan)
).fillna(0)

# Nova coluna corrida suspeita (muito rápido e muito barato)
df_clean['is_suspect_ride'] = (
    (df_clean['speed_mph'] > 50) &
    (df_clean['total_amount'] < 5)
)

# Normalização monetária
mon_cols = [
    'fare_amount', 'extra', 'mta_tax', 'tip_amount', 
    'tolls_amount', 'improvement_surcharge', 'total_amount', 
    'revenue_per_mile', 'revenue_per_min'
]
for cols in mon_cols:
    df_clean[cols] = df_clean[cols].round(2)

# Segmentações
# Faixas de distância (0-3 curta, 3-10 média, 10+ longa)
df_clean['dist_category'] = pd.cut(df_clean['trip_distance'], 
                                   bins=[-np.inf, 3, 10, np.inf], 
                                   labels=['curta', 'media', 'longa'])

# Faixas de duração (0-10 curta, 10-30 normal, 30+ longa)
df_clean['dur_category'] = pd.cut(df_clean['duration_min'], 
                                  bins=[-np.inf, 10, 30, np.inf], 
                                  labels=['curta', 'normal', 'longa'])

# Turnos do dia (0-5 madrugada, 6-11 manhã, 12-17 tarde, 18-23 noite)
df_clean['time_of_day'] = pd.cut(df_clean['hour_of_day'], 
                                 bins=[-1, 5, 11, 17, 23], 
                                 labels=['madrugada', 'manha', 'tarde', 'noite'])

new_cols = ['duration_min', 'speed_mph', 'date', 'hour_of_day', 'day_of_week', 'is_weekend', 'revenue_per_mile', 'revenue_per_min', 'is_suspect_ride', 'dist_category', 'dur_category', 'time_of_day']
print('Novas colunas:')
for col in new_cols:
    print(f'{col}')

Novas colunas:
duration_min
speed_mph
date
hour_of_day
day_of_week
is_weekend
revenue_per_mile
revenue_per_min
is_suspect_ride
dist_category
dur_category
time_of_day


In [10]:
# Remoção colunas originais de datetime (já temos pickup_datetime e dropoff_datetime)
df_clean.drop(columns=['tpep_pickup_datetime', 'tpep_dropoff_datetime'], inplace=True)

# Visualização do resultado final do df
df_clean.head()

,VendorID,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,...,hour_of_day,day_of_week,is_weekend,payment_label,revenue_per_mile,revenue_per_min,is_suspect_ride,dist_category,dur_category,time_of_day
0,2,1,1.59,-73.993896,40.750111,1,N,-73.974785,40.750618,1,...,19,3,False,Credit Card,10.72,0.94,False,curta,normal,noite
1,1,1,3.3,-74.001648,40.724243,1,N,-73.994415,40.759109,1,...,20,5,True,Credit Card,5.39,0.9,False,media,normal,noite
2,1,1,1.8,-73.963341,40.802788,1,N,-73.95182,40.824413,2,...,20,5,True,Cash,6.0,1.07,False,curta,normal,noite
3,1,1,0.5,-74.009087,40.713818,1,N,-74.004326,40.719986,2,...,20,5,True,Cash,9.6,2.57,False,curta,curta,noite
4,1,1,3.0,-73.971176,40.762428,1,N,-74.004181,40.742653,2,...,20,5,True,Cash,5.43,0.84,False,curta,normal,noite


In [12]:
# Agregação por dia
daily_agg = df_clean.groupby('date').agg(
    total_trips=('date', 'size'),
    total_revenue=('total_amount', 'sum'),
    avg_ticket=('total_amount', 'mean'),
    avg_tip=('tip_amount', 'mean')
).round(2)
display(daily_agg.head(10))

,total_trips,total_revenue,avg_ticket,avg_tip
date,,,,
2015-01-01,195687,2883869.37,14.74,1.2
2015-01-02,169741,2412239.35,14.21,1.18
2015-01-03,199668,2717193.95,13.61,1.19
2015-01-04,156793,2337298.91,14.91,1.41
2015-01-05,335331,4850997.42,14.47,1.44
2015-01-06,354660,5016913.22,14.15,1.45
2015-01-07,397100,5482175.56,13.81,1.43
2015-01-08,415263,5840733.53,14.07,1.48
2015-01-09,412884,5947047.09,14.4,1.5


In [ ]:
# Agregação por semana (com variação semanal)
week = pd.to_datetime(df_clean['date']).dt.strftime('%Y-W%V')
weekly_agg = df_clean.groupby(week).agg(
    total_trips=('date', 'size'),
    total_revenue=('total_amount', 'sum'),
    avg_ticket=('total_amount', 'mean'),
    avg_tip=('tip_amount', 'mean')
).fillna(0).round(2)

# Variação semanal
weekly_agg['var_trips_%'] = (weekly_agg['total_trips'].pct_change() * 100).round(2)
weekly_agg['var_revenue_%'] = (weekly_agg['total_revenue'].pct_change() * 100).round(2)
display(weekly_agg.head(10))

,total_trips,total_revenue,avg_ticket,avg_tip,var_trips_%,var_revenue_%
date,,,,,,
2015-W01,721889,10350601.58,14.34,1.24,NaN,<NA>
2015-W02,2778234,39058180.03,14.06,1.44,284.86,277.35
2015-W03,2879864,45012988.36,15.63,2.88,3.66,15.25
2015-W04,2785538,39975978.89,14.35,1.56,-3.28,-11.19
2015-W05,2053734,29405152.41,14.32,1.55,-26.27,-26.44
2016-W01,2384409,34582343.02,14.5,1.62,16.10,17.61
2016-W02,2545982,37690443.21,14.8,1.71,6.78,8.99
2016-W03,2040457,30608908.92,15.0,1.75,-19.86,-18.79
2016-W04,2416618,38408097.9,15.89,1.84,18.44,25.48


In [15]:
# Agregação por vendor
vendor_agg = df_clean.groupby('VendorID').agg(
    total_trips=('date', 'size'),
    total_revenue=('total_amount', 'sum'),
    avg_distance=('trip_distance', 'mean'),
    avg_speed=('speed_mph', 'mean')
).round(2)
vendor_agg

,total_trips,total_revenue,avg_distance,avg_speed
VendorID,,,,
1,19877864,292787710.89,2.66,11.74
2,23896520,364310901.51,2.8,12.17


In [17]:
# Agregação por tipo de pagamento
payment_agg = df_clean.groupby('payment_label').agg(
    total_trips=('payment_label', 'size'),
    total_revenue=('total_amount', 'sum'),
    avg_fare=('fare_amount', 'mean'),
    avg_tip=('tip_amount', 'mean'),
).round(2)
payment_agg['percent_trips'] = ((payment_agg['total_trips'] / payment_agg['total_trips'].sum()) * 100).round(2)
payment_agg

,total_trips,total_revenue,avg_fare,avg_tip,percent_trips
payment_label,,,,,
Cash,15108277,179293445.78,10.66,0.0,34.51
Credit Card,28522537,475681724.53,12.56,2.7,65.16
Dispute,38892,583329.74,13.44,0.0,0.09
No Charge,104675,1540093.75,13.03,0.0,0.24
Unknown,3,18.6,5.33,0.0,0.00


In [19]:
# Agregação por dia da semana
dow_agg = df_clean.groupby('day_of_week').agg(
    total_trips=('day_of_week', 'size'),
    avg_distance=('trip_distance', 'mean'),
    avg_fare=('fare_amount', 'mean'),
    avg_duration=('duration_min', 'mean')
)
days = {
    0: 'Monday',
    1: 'Tuesday',
    2: 'Wednesday',
    3: 'Thursday',
    4: 'Friday',
    5: 'Saturday',
    6: 'Sunday'
}
dow_agg = dow_agg.rename(index=days)
dow_agg

,total_trips,avg_distance,avg_fare,avg_duration
day_of_week,,,,
Monday,5413349,2.824622,11.94321,12.479461
Tuesday,5788737,2.673482,11.909122,13.09763
Wednesday,6275388,2.66289,11.946282,13.263033
Thursday,6780389,2.721965,12.177558,13.562658
Friday,6985159,2.707818,11.997338,13.209403
Saturday,6902561,2.637425,11.42696,12.076774
Sunday,5628801,2.965925,11.955798,11.792161


In [22]:
# Load em parquet
output_dir = path.parent / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'yellow_taxi_dataset.parquet'
relative_path = output_path.relative_to(path.parent)

df_clean.to_parquet(output_path, engine='pyarrow', index=False)
size_mb = output_path.stat().st_size / (1024 ** 3)
print(f"Arquivo: {relative_path}")
print(f"Tamanho: {size_mb:.1f} GB")

Arquivo: data\processed\yellow_taxi_dataset.parquet
Tamanho: 1.4 GB
